# Тишина АЗ-26

## 1. Библиотеки

In [1]:
import sys
import numpy as np
import pandas as pd
import lightgbm as lgb
import sklearn
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('LightGBM:', lgb.__version__)

Python: 3.12.14
NumPy: 2.3.5
pandas: 2.2.3
scikit-learn: 1.8.0
LightGBM: 4.6.0


## 2. Загрузка данных

In [2]:
train = pd.read_csv('telemetry_train.csv')
test = pd.read_csv('telemetry_test.csv')
test_ids = test.timestamp_id.copy()
train = train.sort_values(['session_id', 'timestamp_id']).reset_index(drop=True)
test = test.sort_values(['session_id', 'timestamp_id']).reset_index(drop=True)
sensors = [c for c in test if c not in ['timestamp_id', 'session_id', 'mode', 'dt']]
y = train.is_anomaly.to_numpy()
groups = train.session_id

print('Обучение:', train.shape)
print('Тест:', test.shape)
print(train.head().to_string(index=False))

Обучение: (120000, 20)
Тест: (40000, 19)
 timestamp_id  session_id   mode    dt  t_battery  v_battery  v_bus_aux  i_solar  flux_sensor  t_engine  t_radiator    p_fuel    gyro_x    gyro_y    gyro_z   accel_x   accel_y   accel_z  signal_strength  is_anomaly
       100001        1000 cruise 3.960    18.4992   27.74546   12.04274   4.5798     1409.751   35.4953     44.5563 2083148.0 -0.000481 -0.002372  0.000826 -0.000044  0.001684 -0.007858            -92.0           0
       100002        1000 cruise 4.040    18.6014   27.76955   12.02893   4.5783     1403.711   35.6737     44.5217 2081503.0 -0.000432 -0.001176 -0.000974  0.001071  0.001302 -0.003225            -92.5           0
       100003        1000 cruise 4.044    18.6723   27.79677   12.00896   4.5326     1406.552   35.7162     44.4160 2080716.0 -0.000111 -0.001229 -0.001677  0.002720 -0.000717 -0.007762            -92.5           0
       100004        1000 cruise 4.019    18.6732   27.71568   11.98545   4.4637     1409.564   35.

## 3. Пропуски, режимы и сеансы

In [3]:
missing = pd.DataFrame({
    'train_missing_pct': train[sensors].isna().mean() * 100,
    'test_missing_pct': test[sensors].isna().mean() * 100
})
print(missing.round(2).to_string())
print()
print(train.groupby('mode').is_anomaly.agg(['size', 'sum', 'mean']).to_string())
print()
print('Сеансов в обучении:', train.session_id.nunique())
print('Сеансов в тесте:', test.session_id.nunique())
print('Общих сеансов:', len(set(train.session_id) & set(test.session_id)))
print('Доля аномалий:', y.mean())

runs = (train.is_anomaly.ne(train.is_anomaly.shift()) |
        train.session_id.ne(train.session_id.shift())).cumsum()
episodes = train.groupby(runs).agg(label=('is_anomaly', 'first'), length=('is_anomaly', 'size'))
episodes = episodes.loc[episodes.label == 1, 'length']
print('Число эпизодов:', len(episodes))
print(episodes.describe().to_string())

                 train_missing_pct  test_missing_pct
t_battery                     3.00              3.02
v_battery                     2.97              3.00
v_bus_aux                     2.95              3.03
i_solar                       2.96              3.00
flux_sensor                   2.99              2.98
t_engine                      2.97              2.95
t_radiator                    2.97              3.02
p_fuel                        2.99              3.02
gyro_x                        2.98              3.01
gyro_y                        2.94              3.00
gyro_z                        2.99              3.00
accel_x                       2.95              2.88
accel_y                       2.96              2.96
accel_z                       2.95              2.94
signal_strength               2.95              2.97

           size   sum      mean
mode                           
cruise    43124  2069  0.047978
eclipse   41839  2040  0.048758
maneuver  35037  1589  

## 4. Опорное решение

In [4]:
baseline_score = np.zeros(len(train))
for c in sensors:
    values = train[c].ffill().fillna(0)
    usual = values.groupby(train.session_id).transform('median')
    baseline_score += (values - usual).abs() / (values.std() + 1e-9)
baseline_score /= len(sensors)
baseline_auc = roc_auc_score(y, baseline_score)
print(f'ROC-AUC baseline на обучающем архиве: {baseline_auc:.6f}')

ROC-AUC baseline на обучающем архиве: 0.530911


## 5. Связи датчиков внутри полного сеанса без использования меток

In [5]:
def relation_features(df):
    sensors = [c for c in df if c not in ['timestamp_id','session_id','mode','dt','is_anomaly']]
    parts=[]
    for sid,d in df.groupby('session_id',sort=False):
        x=d[sensors].interpolate(limit_direction='both')
        good=d[sensors].notna().all(axis=1)
        stable=d['mode'].eq(d['mode'].shift())
        centered=x-x.groupby(d['mode']).transform('median')
        reps={'level':centered,'diff1':x.diff(),'diff3':x.diff(3),'local':x-x.rolling(101,center=True,min_periods=1).median()}
        f={}
        for name,rep in reps.items():
            scale=rep.std().to_numpy()
            z=(rep/scale).fillna(0).to_numpy()
            fit=good & stable
            if name.startswith('diff'):
                lag=int(name[-1]);fit=fit & good.shift(lag,fill_value=False) & d['mode'].eq(d['mode'].shift(lag))
            a=z[fit]
            mu=np.median(a,axis=0)
            cov=np.cov(a,rowvar=False)
            for _ in range(4):
                precision=np.linalg.inv(cov+np.diag(np.diag(cov))*.015)
                dist=np.einsum('ij,jk,ik->i',a-mu,precision,a-mu)
                keep=dist<np.quantile(dist,.85)
                mu=a[keep].mean(axis=0)
                cov=np.cov(a[keep],rowvar=False)
            precision=np.linalg.inv(cov+np.diag(np.diag(cov))*.015)
            residual=(z-mu)@precision/np.sqrt(np.diag(precision))
            q=np.einsum('ij,jk,ik->i',z-mu,precision,z-mu)
            f[name+'_mahal']=q
            f[name+'_relative_mahal']=q/(np.square(z-mu).sum(axis=1)+.1)
            f[name+'_max_residual']=np.abs(residual).max(axis=1)
            for j,c in enumerate(sensors):
                f[name+'_'+c+'_conditional']=np.abs(residual[:,j])
            vals,vec=np.linalg.eigh(cov)
            pc=(z-mu)@vec/np.sqrt(np.maximum(vals,.005))
            for k in [1,2,3,5,8]:
                f[name+f'_pca_{k}']=np.square(pc[:,:k]).mean(axis=1)
        base=pd.DataFrame(f,index=d.index)
        f=[base]
        for w in [5,15,31]:
            f.append(base.rolling(w,center=True,min_periods=1).mean().add_suffix(f'_mean_{w}'))
        summary=base[[c for c in base if any(t in c for t in ['mahal','max_residual','pca_3'])]]
        for lag in [-30,-15,15,30]:
            f.append(summary.rolling(15,center=True,min_periods=1).mean().shift(lag).add_suffix(f'_shift_{lag}'))
        parts.append(pd.concat(f,axis=1).astype('float32'))
    return pd.concat(parts).replace([np.inf,-np.inf],np.nan)

## 6. Быстрые и медленные изменения датчиков

In [6]:
def spectral_features(df):
    sensors=[c for c in df if c not in ['timestamp_id','session_id','mode','dt','is_anomaly']]
    parts=[]
    for sid,d in df.groupby('session_id',sort=False):
        x=d[sensors].interpolate(limit_direction='both')
        x=x-x.groupby(d['mode']).transform('median')
        stable=d['mode'].eq(d['mode'].shift(31)) & d['mode'].eq(d['mode'].shift(-31))
        f={}
        for w in [5,15,31]:
            smooth=x.rolling(w,center=True,min_periods=1).mean()
            for name,rep in [('low',smooth),('high',x-smooth)]:
                z=(rep/rep.std()).fillna(0).to_numpy()
                a=z[stable];mu=np.median(a,axis=0);cov=np.cov(a,rowvar=False)
                for _ in range(4):
                    prec=np.linalg.inv(cov+np.diag(np.diag(cov))*.003)
                    dist=np.einsum('ij,jk,ik->i',a-mu,prec,a-mu)
                    keep=dist<np.quantile(dist,.9)
                    mu=a[keep].mean(axis=0);cov=np.cov(a[keep],rowvar=False)
                prec=np.linalg.inv(cov+np.diag(np.diag(cov))*.003)
                res=(z-mu)@prec/np.sqrt(np.diag(prec))
                vals,vec=np.linalg.eigh(cov)
                pc=(z-mu)@vec/np.sqrt(np.maximum(vals,.001))
                f[f'{name}_{w}_mahal']=np.einsum('ij,jk,ik->i',z-mu,prec,z-mu)
                for k in [1,2,3,5,8]:
                    f[f'{name}_{w}_pca_{k}']=np.square(pc[:,:k]).mean(axis=1)
                r=np.sort(np.square(res),axis=1)
                for k in [1,2,3,5,10]:
                    f[f'{name}_{w}_top_{k}']=r[:,-k:].mean(axis=1)
        base=pd.DataFrame(f,index=d.index)
        ff=[base]
        for w in [5,15,31]:
            ff.append(base.rolling(w,center=True,min_periods=1).mean().add_suffix(f'_mean_{w}'))
        r=pd.concat(ff,axis=1)
        ranks=r.rank(pct=True).add_suffix('_rank')
        parts.append(pd.concat([r,ranks],axis=1).astype('float32'))
    return pd.concat(parts)

## 7. Признаки обучения и теста

In [7]:
R = relation_features(train.drop(columns='is_anomaly'))
RT = relation_features(test)
S = spectral_features(train.drop(columns='is_anomaly'))
ST = spectral_features(test)

features = {
    'relations': (R, RT),
    'multiscale': (pd.concat([R, S], axis=1), pd.concat([RT, ST], axis=1))
}
for name, (X, X_test) in features.items():
    print(name, X.shape, X_test.shape)

relations (120000, 432) (40000, 432)
multiscale (120000, 960) (40000, 960)


## 8. Валидация на четырёх непересекающихся наборах сеансов

In [8]:
folds = list(GroupKFold(n_splits=4).split(R, y, groups))
oof = {}
test_predictions = {}
results = []
importance_rows = []

for name, (X, X_test) in features.items():
    oof[name] = np.zeros(len(train))
    test_predictions[name] = np.zeros(len(test))
    for fold, (train_index, valid_index) in enumerate(folds):
        model = lgb.LGBMClassifier(
            objective='binary',
            metric='auc',
            n_estimators=1800,
            learning_rate=0.025,
            num_leaves=15 if name == 'relations' else 7,
            min_child_samples=120,
            colsample_bytree=0.85,
            reg_lambda=10,
            verbosity=-1,
            n_jobs=6,
            random_state=42 + fold,
            deterministic=True,
            force_col_wise=True
        )
        model.fit(
            X.iloc[train_index], y[train_index],
            eval_set=[(X.iloc[valid_index], y[valid_index])],
            callbacks=[lgb.early_stopping(160, verbose=False)]
        )
        oof[name][valid_index] = model.predict_proba(X.iloc[valid_index])[:, 1]
        test_predictions[name] += model.predict_proba(X_test)[:, 1] / len(folds)
        auc = roc_auc_score(y[valid_index], oof[name][valid_index])
        results.append([name, fold + 1, model.best_iteration_, auc])
        importance = pd.Series(model.feature_importances_, index=X.columns, dtype=float)
        importance_rows.append(importance / importance.sum())
        valid_sessions = sorted(train.iloc[valid_index].session_id.unique().tolist())
        print(f'{name}, fold {fold + 1}, iterations {model.best_iteration_}, ROC-AUC {auc:.6f}')
        print('Сеансы валидации:', valid_sessions)
    print(f'{name}, общий OOF ROC-AUC: {roc_auc_score(y, oof[name]):.6f}')
    print()

cv_results = pd.DataFrame(results, columns=['model', 'fold', 'iterations', 'roc_auc'])
print(cv_results.to_string(index=False))

relations, fold 1, iterations 1051, ROC-AUC 0.782987
Сеансы валидации: [1006, 1008, 1017, 1021, 1022, 1024, 1025, 1029]
relations, fold 2, iterations 47, ROC-AUC 0.682484
Сеансы валидации: [1005, 1009, 1010, 1013, 1026, 1028, 1030, 1031]
relations, fold 3, iterations 26, ROC-AUC 0.733525
Сеансы валидации: [1000, 1001, 1004, 1007, 1011, 1012, 1015, 1019]
relations, fold 4, iterations 130, ROC-AUC 0.766122
Сеансы валидации: [1002, 1003, 1014, 1016, 1018, 1020, 1023, 1027]
relations, общий OOF ROC-AUC: 0.734060

multiscale, fold 1, iterations 8, ROC-AUC 0.779312
Сеансы валидации: [1006, 1008, 1017, 1021, 1022, 1024, 1025, 1029]
multiscale, fold 2, iterations 113, ROC-AUC 0.683014
Сеансы валидации: [1005, 1009, 1010, 1013, 1026, 1028, 1030, 1031]
multiscale, fold 3, iterations 37, ROC-AUC 0.746028
Сеансы валидации: [1000, 1001, 1004, 1007, 1011, 1012, 1015, 1019]
multiscale, fold 4, iterations 137, ROC-AUC 0.768810
Сеансы валидации: [1002, 1003, 1014, 1016, 1018, 1020, 1023, 1027]
multisca

## 9. Сглаживание внутри сеансов и ансамбль

In [9]:
def session_rank(values, session_ids):
    values = pd.Series(np.asarray(values), index=session_ids.index)
    return values.groupby(session_ids).rank(pct=True).to_numpy()

def smooth_rank(values, session_ids):
    values = pd.Series(values, index=session_ids.index)
    values = values.groupby(session_ids).transform(
        lambda part: part.rolling(11, center=True, min_periods=1).mean()
    )
    return session_rank(values, session_ids)

stat_train = (
    session_rank(R['diff3_pca_5_mean_15'], train.session_id) +
    S['high_31_pca_5_mean_15_rank'].to_numpy()
) / 2
stat_test = (
    session_rank(RT['diff3_pca_5_mean_15'], test.session_id) +
    ST['high_31_pca_5_mean_15_rank'].to_numpy()
) / 2

score_oof = (
    0.375 * smooth_rank(oof['relations'], train.session_id) +
    0.375 * smooth_rank(oof['multiscale'], train.session_id) +
    0.25 * stat_train
)
score_test = (
    0.375 * smooth_rank(test_predictions['relations'], test.session_id) +
    0.375 * smooth_rank(test_predictions['multiscale'], test.session_id) +
    0.25 * stat_test
)

rows = []
for name in features:
    rows.append([name, roc_auc_score(y, oof[name])])
    rows.append([name + '_smooth_rank', roc_auc_score(y, smooth_rank(oof[name], train.session_id))])
rows.append(['statistical', roc_auc_score(y, stat_train)])
rows.append(['ensemble', roc_auc_score(y, score_oof)])
print(pd.DataFrame(rows, columns=['model', 'roc_auc']).to_string(index=False))
print()
for fold, (_, valid_index) in enumerate(folds):
    print(f'Ансамбль, fold {fold + 1}: {roc_auc_score(y[valid_index], score_oof[valid_index]):.6f}')

                 model  roc_auc
             relations 0.734060
 relations_smooth_rank 0.744373
            multiscale 0.721425
multiscale_smooth_rank 0.747906
           statistical 0.738135
              ensemble 0.754480

Ансамбль, fold 1: 0.803932
Ансамбль, fold 2: 0.686191
Ансамбль, fold 3: 0.758650
Ансамбль, fold 4: 0.770974


## 10. Значимость признаков

In [10]:
importance = pd.concat(importance_rows, axis=1).fillna(0).mean(axis=1)
importance = importance.sort_values(ascending=False).head(20)
print(importance.to_string())

high_31_mahal_mean_15_rank               0.023881
level_mahal_shift_30                     0.016561
level_p_fuel_conditional_mean_31         0.015012
local_mahal                              0.014989
high_31_pca_5_mean_15_rank               0.014856
local_pca_8_mean_31                      0.014613
diff1_flux_sensor_conditional_mean_31    0.014419
diff3_flux_sensor_conditional_mean_31    0.014399
level_v_battery_conditional_mean_31      0.013995
high_5_pca_2_mean_31                     0.012620
diff1_pca_8                              0.012049
level_pca_5_mean_5                       0.011721
level_t_battery_conditional_mean_31      0.011528
local_mahal_mean_31                      0.011507
level_pca_8_mean_15                      0.010093
diff3_mahal_mean_31                      0.009861
level_relative_mahal_mean_15             0.009791
local_t_battery_conditional_mean_31      0.009056
low_31_pca_2_mean_31_rank                0.008757
diff3_pca_5_mean_31                      0.008743


## 11. Калибровка оценок по OOF-прогнозам

In [11]:
def logit_feature(score):
    score = np.clip(score, 1e-5, 1 - 1e-5)
    return np.log(score / (1 - score)).reshape(-1, 1)

calibrator = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
calibrator.fit(logit_feature(score_oof), y)
anomaly_prob = calibrator.predict_proba(logit_feature(score_test))[:, 1]
print('Коэффициент калибровки:', calibrator.coef_[0, 0])
print(pd.Series(anomaly_prob, name='anomaly_prob').describe().to_string())

Коэффициент калибровки: 0.7472434902272794
count    40000.000000
mean         0.049261
std          0.072343
min          0.000347
25%          0.013400
50%          0.026449
75%          0.051625
max          0.771551


## 12. Сохранение ответа

In [12]:
submission = pd.DataFrame({
    'timestamp_id': test.timestamp_id,
    'anomaly_prob': anomaly_prob
})
submission = submission.set_index('timestamp_id').loc[test_ids].reset_index()
submission.to_csv('submission.csv', index=False, float_format='%.12f')
print(submission.head(10).to_string(index=False))
print('Сохранён submission.csv:', submission.shape)
print(f'OOF ROC-AUC: {roc_auc_score(y, score_oof):.6f}')

 timestamp_id  anomaly_prob
       220001      0.004034
       220002      0.003939
       220003      0.003893
       220004      0.003829
       220005      0.003584
       220006      0.003375
       220007      0.003919
       220008      0.004712
       220009      0.005444
       220010      0.006465
Сохранён submission.csv: (40000, 2)
OOF ROC-AUC: 0.754480
